# Llama-3.1-8B-Instruct in 8-bit bitsandbytes

Let's first install the required libraries:

In [ ]:
! pip install transformers[torch] bitsandbytes

We import the required libraries : 

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig  

Let's load the model. To quantize the model on the fly, we pass a `quantization_config`:

In [ ]:
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

quantized_model = AutoModelForCausalLM.from_pretrained(
	model_name, device_map="auto", torch_dtype=torch.bfloat16, quantization_config=quantization_config)

Then, we need to prepare the inputs: 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
input_text = "What are we having for dinner?"
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")

Finally, we can generate the output ! 

In [ ]:
output = quantized_model.generate(**input_ids, max_new_tokens=10)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Save the 8-bit quantized model locally:

In [ ]:
from pathlib import Path

OUTPUT_DIR = "/home/spark/projects/stoic/output/vllm/llama31_8b_instruct_8bit"

print(f"💾 Saving 8-bit model to {OUTPUT_DIR}...")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

quantized_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ 8-bit model saved!")
print(f"\n🚀 To serve with vLLM:")
print(f"   vllm serve {OUTPUT_DIR} --quantization bitsandbytes --load-format bitsandbytes")